In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

In [2]:
cols = [
    "age", "sex", "on_thyroxine", "query_on_thyroxine", "on_antithyroid_medication",
    "sick", "pregnant", "thyroid_surgery", "I131_treatment", "query_hypothyroid",
    "query_hyperthyroid", "lithium", "goitre", "tumor", "hypopituitary", "psych",
    "TSH_measured", "TSH", "T3_measured", "T3", "TT4_measured", "TT4",
    "T4U_measured", "T4U", "FTI_measured", "FTI", "TBG_measured", "TBG",
    "referral_source", "class_raw",
]
df = pd.read_csv("../data/thyroid0387.data", header=None, names=cols, na_values="?")

df["code"] = df["class_raw"].str.replace(r"\[[0-9]+\]", "", regex=True)
def map_category(code):
    if code == "-": return "Negative"
    if code in ("A", "B", "C", "D"): return "Hyperthyroid"
    if code == "F": return "Primary Hypothyroid"
    if code == "G": return "Compensated Hypothyroid"
    return "Other Thyroid Condition"
df["target"] = df["code"].apply(map_category)
print("Loaded:", df.shape)

Loaded: (9172, 32)


In [3]:
drop_cols = [
    "TBG", "TBG_measured",
    "TSH_measured", "T3_measured", "TT4_measured", "T4U_measured", "FTI_measured",
    "referral_source", "class_raw", "code",
]
df = df.drop(columns=drop_cols)
print("Remaining columns:", df.shape[1])
df.columns.tolist()

Remaining columns: 22


['age',
 'sex',
 'on_thyroxine',
 'query_on_thyroxine',
 'on_antithyroid_medication',
 'sick',
 'pregnant',
 'thyroid_surgery',
 'I131_treatment',
 'query_hypothyroid',
 'query_hyperthyroid',
 'lithium',
 'goitre',
 'tumor',
 'hypopituitary',
 'psych',
 'TSH',
 'T3',
 'TT4',
 'T4U',
 'FTI',
 'target']

In [4]:
print("Ages > 120 before fix:", (df["age"] > 120).sum())
df.loc[df["age"] > 120, "age"] = np.nan

Ages > 120 before fix: 4


In [5]:
binary_cols = [
    "on_thyroxine", "query_on_thyroxine", "on_antithyroid_medication", "sick",
    "pregnant", "thyroid_surgery", "I131_treatment", "query_hypothyroid",
    "query_hyperthyroid", "lithium", "goitre", "tumor", "hypopituitary", "psych",
]
for col in binary_cols:
    df[col] = df[col].map({"t": 1, "f": 0})

df["sex"] = df["sex"].map({"M": 1, "F": 0})
print("sex missing after mapping:", df["sex"].isna().sum())

sex missing after mapping: 307


In [6]:
continuous = ["age", "TSH", "T3", "TT4", "T4U", "FTI"]
for col in continuous:
    df[col] = df[col].fillna(df[col].median())

df["sex"] = df["sex"].fillna(df["sex"].mode()[0])

print("Total missing values remaining:", df.isna().sum().sum())

Total missing values remaining: 0


In [7]:
le = LabelEncoder()
y = le.fit_transform(df["target"])
X = df.drop(columns=["target"])

print("Features:", X.shape[1])
print("Classes:", list(le.classes_))
print("Encoded as:", dict(zip(le.classes_, le.transform(le.classes_))))

Features: 21
Classes: ['Compensated Hypothyroid', 'Hyperthyroid', 'Negative', 'Other Thyroid Condition', 'Primary Hypothyroid']
Encoded as: {'Compensated Hypothyroid': np.int64(0), 'Hyperthyroid': np.int64(1), 'Negative': np.int64(2), 'Other Thyroid Condition': np.int64(3), 'Primary Hypothyroid': np.int64(4)}


In [8]:
smote = SMOTE(random_state=42, k_neighbors=1)
X_balanced, y_balanced = smote.fit_resample(X, y)

print("Before SMOTE:", X.shape[0], "records")
print("After  SMOTE:", X_balanced.shape[0], "records")
print("Per-class counts:", np.bincount(y_balanced))

Before SMOTE: 9172 records
After  SMOTE: 33855 records
Per-class counts: [6771 6771 6771 6771 6771]


In [9]:
processed = X_balanced.copy()
processed["target_encoded"] = y_balanced
processed.to_csv("../data/processed_balanced.csv", index=False)

joblib.dump(le, "../models/label_encoder.pkl")
joblib.dump(X.columns.tolist(), "../models/feature_columns.pkl")

print("Saved processed_balanced.csv", processed.shape)
print("Saved label_encoder.pkl and feature_columns.pkl")

Saved processed_balanced.csv (33855, 22)
Saved label_encoder.pkl and feature_columns.pkl
